# Iterative probing — how many dimensions of `h` carry a linear position code?

**Thread:** `editability/` · **Metric/editor definitions:** `../METRICS_AND_EDITORS.md` · **Conventions:**
`../../../../CLAUDE.md`. **Model:** the standard GRU `runs/controls/H256` (H=256, affine decoder) — the exact
checkpoint the editability findings were established on. **Dataset:** `datasets/4_fixed_refl_inview`, `test`
split. **No retraining**; only linear/MLP probes are fit here.

## The question

A single linear position probe `p = A h + b` reads position at R² ≈ 0.82, and its **row space is only 4 of the
256 dimensions** of `h`. That 4-dimensional slice is the entire reachable set of a readout-injection editor,
which is why §4 of the thread reports its row-space fraction against a `√(4/256)` chance level.

But 4 dimensions is the dimensionality of *one* probe, not of the position code. If position is written
redundantly across many directions, a probe reading one 4-dim slice would be blind to the rest — and "the
position subspace" would be much larger than the object we have been measuring against.

So: **fit a probe, delete the subspace it reads, and see whether position is still there.** Repeat until it
is not.

## The procedure

This is **Iterative Nullspace Projection** (INLP, Ravfogel et al. 2020), used here to *measure* the
dimensionality of a linear code rather than to erase an attribute:

1. Fit a least-squares probe `A₁ : h → (x₀, y₀, x₁, y₁)` on the state bank.
2. Take an orthonormal basis `B₁` of `row(A₁)` (its rank, generically 4) and **project it out**:
   `h ← h − B₁B₁ᵀh`. The probe is now exactly blind: any `A₁`-readable content is gone.
3. Fit a fresh probe `A₂` on the deflated states. Repeat.
4. Stop when held-out R² reaches the chance floor.

**Why the accumulated subspaces are orthogonal, and why the total is `Σ rank`.** At step `k` the probe is fit
on states that already lie in the remaining subspace. `np.linalg.lstsq` returns the **minimum-norm** solution,
whose rows therefore lie in the row space of the (deflated) design matrix — i.e. inside the remaining subspace,
orthogonal to everything already removed. So the accumulated directions form an orthonormal basis and the
total dimension is the **sum of the per-probe ranks**. Sevan's `4 × (number of probes)` is right *provided*
every probe is full-rank-4; cell [3] **asserts both** the orthogonality and the rank rather than assuming them,
and the reported total is the measured `Σ rank`.

## What could make this misleading, and the controls for each

| worry | control |
|---|---|
| R² falls because we deleted **variance**, not because we deleted **position** | a **random-subspace ablation** track that removes the *same number* of dimensions per step, chosen at random from the remaining subspace |
| "near chance" is asserted rather than measured | a **shuffled-label floor**: the same probe fit against row-permuted targets, which is the R² attainable by fitting noise |
| dimensions are counted against 256, but the states do not fill 256 dimensions | the **PCA hull** of the visited-state bank (dims to 90 / 95 / 99% variance) as the honest denominator |
| a single "the dimensionality is N" hides a gradual decay | the whole **R² curve**, plus dimension reported at several R² thresholds |
| position might survive **nonlinearly** after its linear directions are gone | an **MLP probe** refit on the residual states at every iteration |

## Definitions — read this before any number

| term | meaning |
|---|---|
| **state bank** | all aligned hidden states `h` from `NPROBE` held-out `test` sequences, one per (sequence, timestep). Alignment: `decode(h_t) ↔ obs[t+1]`, matching the states the editability thread edits. |
| **probe** | least-squares `A h + b` with `A` of shape `(4, 256)`; target is `(x₀, y₀, x₁, y₁)` in sim units |
| **row space** `row(A)` | span of `A`'s rows — the only part of `h` the probe can see *or* move. Dimension = `rank(A)` ≤ 4. |
| **deflation step** | `h ← h − BBᵀh`, `B` an orthonormal basis of `row(A)`. After it, that probe reads exactly a constant. |
| **iteration `k`** | the `k`-th probe, fit on states with all `k−1` previous row spaces removed |
| **accumulated dimension** | `Σ_{j≤k} rank(A_j)` — the size of the removed subspace after `k` iterations |

| metric | formula | units | better |
|---|---|---|---|
| **position R² (held-out)** | `1 − ‖Y − (Ah+b)‖²/‖Y − Ȳ‖²` on the held-out 20% of **sequences**, probe fit on the other 80% | — | ↑ |
| **position RMSE** | `√mean‖Y − (Ah+b)‖²` on the held-out split | sim units | ↓ |
| **variance remaining** | `‖H_k‖²_F / ‖H_0‖²_F`, the fraction of total state energy left after `k` deflations | fraction | — |
| **shuffled-label floor** | the same held-out R², with target rows randomly permuted before fitting | — | — |
| **PCA hull @ p%** | number of PCA components of the *undeflated* bank reaching ≥ p% of its variance | dims | — |

### Three things to hold in mind while reading

1. **The chance level for a held-out R² is 0**, not `1/√H` or anything dimensional — a probe that has learned
   nothing predicts the training mean and scores 0. The shuffled-label track measures where that floor actually
   sits at this sample size rather than asserting it.
2. **The split is by SEQUENCE, not by row.** Consecutive frames of one sequence are near-duplicates; a random
   row split would leak them across the boundary and inflate every R² on the page.
3. **This measures the dimensionality of the *greedily-deflated linear* position code.** It is a property of
   the procedure as well as of the model: a different removal order could in principle exhaust position in a
   different number of steps, and a gradual R² decay means "the" dimension depends on where you put the
   threshold. Hence the curve and a threshold table rather than a single headline number.

In [ ]:
# [1] Setup: load the standard GRU, build the aligned state bank, and fix the split by SEQUENCE.
import os, sys, time
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.figures.theme import style_ax

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ = 2
NPROBE = 2000          # test sequences in the bank
HOLDOUT = 0.2          # held-out fraction, split by SEQUENCE
MAX_ITER = 50          # hard cap on deflation steps
R2_STOP = 0.02         # stop once held-out R² falls below this
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

ROOT = "../../../.."
CKPT = f"{ROOT}/runs/controls/H256/best_model.pt"
MODEL, INFO = load_checkpoint(CKPT, device=DEVICE)
bundle = load_dataset(f"{ROOT}/datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ, require_edits=False)
test = bundle.test
Hd = MODEL.hidden_size

@torch.no_grad()
def aligned_bank(model, obs_np, chunk=500):
    """States aligned so decode(bank[:, t]) ↔ sim frame t+1 — the same states the thread edits."""
    outs = []
    for s in range(0, len(obs_np), chunk):
        o = torch.from_numpy(obs_np[s:s + chunk]).float().to(DEVICE)
        state = None; seq = []
        for t in range(o.shape[1] - 1):
            _, state = model.step(o[:, t], state)
            seq.append(model.flat_state(state))
        outs.append(torch.stack(seq, 1).cpu().numpy())
    return np.concatenate(outs, 0)

t0 = time.perf_counter()
obs_pr = test.obs[:NPROBE].astype(np.float32)
BANK = aligned_bank(MODEL, obs_pr)                                     # (N, T-1, Hd)
T_use = BANK.shape[1]
POS = test.positions[:NPROBE, 1:1 + T_use, :N_OBJ, :].reshape(NPROBE, T_use, N_OBJ * 2)

# float64 throughout: 40 successive projections in float32 accumulate enough error to
# break the orthogonality this whole measurement rests on.
H_ALL = BANK.reshape(-1, Hd).astype(np.float64)
Y_ALL = POS.reshape(-1, N_OBJ * 2).astype(np.float64)

n_tr_seq = int((1 - HOLDOUT) * NPROBE)
TR = slice(0, n_tr_seq * T_use)                                        # by SEQUENCE, not by row
TE = slice(n_tr_seq * T_use, None)
print(f"bank built in {time.perf_counter()-t0:.0f}s: {H_ALL.shape[0]:,} states x {Hd} dims "
      f"({NPROBE} sequences x {T_use} timesteps)")
print(f"split by sequence: {n_tr_seq} train / {NPROBE - n_tr_seq} held out "
      f"-> {H_ALL[TR].shape[0]:,} / {H_ALL[TE].shape[0]:,} rows")
print(f"model: {INFO.run_name}, H={Hd}, val loss {INFO.val_loss:.5f}")

# PCA hull of the UNDEFLATED bank — the honest denominator for "how many dimensions".
Hc = H_ALL - H_ALL.mean(0)
sv = np.linalg.svd(Hc[::7], compute_uv=False)                          # subsample rows; spectrum is stable
evr = np.cumsum(sv ** 2) / (sv ** 2).sum()
HULL = {p: int(np.searchsorted(evr, p) + 1) for p in (0.90, 0.95, 0.99)}
print(f"PCA hull of the visited states: {HULL[0.90]} dims @90%, {HULL[0.95]} @95%, {HULL[0.99]} @99% "
      f"(of {Hd} ambient)")

In [ ]:
# [2] The two primitives: a least-squares probe, and one deflation step.
def fit_probe(X, Y, tr, te):
    """Least-squares probe on X[tr] -> Y[tr], scored on X[te]. Returns A (d, D), b, and held-out fit.

    np.linalg.lstsq returns the MINIMUM-NORM solution, which is what makes the accumulated
    row spaces orthogonal: A's rows land in the row space of the (already deflated) design
    matrix, hence inside the remaining subspace."""
    Aug = np.concatenate([X[tr], np.ones((X[tr].shape[0], 1))], 1)
    sol, *_ = np.linalg.lstsq(Aug, Y[tr], rcond=None)
    W, b = sol[:-1], sol[-1]                                  # W: (D, d)
    pred = X[te] @ W + b
    ss_res = ((pred - Y[te]) ** 2).sum()
    ss_tot = ((Y[te] - Y[tr].mean(0)) ** 2).sum()             # baseline = the TRAIN mean, not the test mean
    return dict(A=W.T, b=b,
                r2=float(1 - ss_res / ss_tot),
                rmse=float(np.sqrt(((pred - Y[te]) ** 2).sum(1).mean())))

def rowspace_basis(A, tol=1e-8):
    """Orthonormal basis of row(A) as columns, plus its numerical rank."""
    _, s, vt = np.linalg.svd(A, full_matrices=False)
    rank = int((s > s[0] * tol).sum()) if s.size and s[0] > 0 else 0
    return vt[:rank].T, rank                                  # (D, rank)

def deflate(X, B):
    """Remove span(B) from every row of X."""
    return X - (X @ B) @ B.T

def random_basis(remaining_B, k, rng):
    """k random orthonormal directions drawn from INSIDE the current remaining subspace.

    Matched to the real track: it removes the same number of dimensions from the same space,
    so the only difference is whether those dimensions were chosen to carry position."""
    G = rng.standard_normal((remaining_B.shape[1], k))
    Q, _ = np.linalg.qr(G)
    return remaining_B @ Q

print("primitives ready")

In [ ]:
# [3] Run the deflation, with the random-ablation control and the shuffled-label floor in lockstep.
rng = np.random.default_rng(0)
Y_SHUF = Y_ALL.copy(); rng.shuffle(Y_SHUF)          # break the h <-> position correspondence

# state of the three tracks
X_real = H_ALL.copy()
X_ctrl = H_ALL.copy()
REM = np.eye(Hd)                                    # orthonormal basis of the control's remaining subspace
DIRS = []                                           # accumulated removed directions, real track
CASCADE = []                                        # the probes themselves, needed by the editor below
E0 = (H_ALL ** 2).sum()

ROWS = []
t0 = time.perf_counter()
for k in range(1, MAX_ITER + 1):
    p_real = fit_probe(X_real, Y_ALL, TR, TE)
    p_ctrl = fit_probe(X_ctrl, Y_ALL, TR, TE)
    p_shuf = fit_probe(X_real, Y_SHUF, TR, TE)

    B, rank = rowspace_basis(p_real["A"])
    if rank == 0:
        print(f"iteration {k}: probe is rank 0 — nothing left to remove; stopping."); break

    # ASSERT the two properties the "4 x number of probes" arithmetic depends on.
    if DIRS:
        prev = np.concatenate(DIRS, 1)
        leak = float(np.abs(prev.T @ B).max())
        assert leak < 1e-6, f"iteration {k}: new basis not orthogonal to removed subspace (max |inner| {leak:.2e})"
    orth = float(np.abs(B.T @ B - np.eye(rank)).max())
    assert orth < 1e-9, f"iteration {k}: basis not orthonormal ({orth:.2e})"

    dim_before = sum(r["rank"] for r in ROWS)      # dims already removed when THIS probe was fit
    ROWS.append(dict(k=k, rank=rank, dim_before=dim_before, dim=dim_before + rank,
                     r2=p_real["r2"], rmse=p_real["rmse"],
                     r2_ctrl=p_ctrl["r2"], rmse_ctrl=p_ctrl["rmse"], r2_shuf=p_shuf["r2"],
                     var_real=float((X_real ** 2).sum() / E0), var_ctrl=float((X_ctrl ** 2).sum() / E0)))

    DIRS.append(B)
    CASCADE.append(dict(A=p_real['A'], b=p_real['b'], r2=p_real['r2'], B=B))
    X_real = deflate(X_real, B)
    Bc = random_basis(REM, rank, rng)                # matched random removal from the remaining space
    X_ctrl = deflate(X_ctrl, Bc)
    REMp = REM - Bc @ (Bc.T @ REM)                   # shrink the control's remaining subspace...
    U_, s_, _ = np.linalg.svd(REMp, full_matrices=False)
    REM = U_[:, s_ > s_[0] * 1e-10]                  # ...and re-orthonormalise, dropping the killed directions

    if p_real["r2"] < R2_STOP:
        print(f"iteration {k}: held-out R² {p_real['r2']:.4f} < {R2_STOP} — position exhausted; stopping.")
        break

N_IT = len(ROWS)
# "the code is N dims" means: after removing N dims, no meaningful linear position signal is left.
# That is the LAST probe's dim_before — the state it was actually fit on.
TOTAL_DIM = ROWS[-1]["dim_before"]
print(f"\nran {N_IT} iterations in {time.perf_counter()-t0:.0f}s")
print(f"every probe rank: {sorted(set(r['rank'] for r in ROWS))}  -> "
      + ("all rank 4, so total = 4 x n_probes" if set(r['rank'] for r in ROWS) == {4}
         else "NOT all rank 4 — the total is the measured sum of ranks, not 4 x n_probes"))
print(f"linear position code exhausted after removing {TOTAL_DIM} dims of {Hd} ambient "
      f"({100*TOTAL_DIM/Hd:.1f}%), = {100*TOTAL_DIM/HULL[0.99]:.1f}% of the {HULL[0.99]}-dim 99% PCA hull")
print(f"orthogonality of the accumulated basis verified at every step (max |inner product| < 1e-6)")

In [ ]:
# [4] Table 1 + Fig 1 — the decay, against the two controls and the variance actually removed.
rows = ["| iteration | rank | dims removed BEFORE this probe | position R² (held-out) | position RMSE (sim units) | R² random-ablation control | R² shuffled-label floor | variance remaining (real) | variance remaining (control) |",
        "|---|---|---|---|---|---|---|---|---|"]
for r in ROWS:
    rows.append(f"| {r['k']} | {r['rank']} | {r['dim_before']} | **{r['r2']:.3f}** | {r['rmse']:.3f} | "
                f"{r['r2_ctrl']:.3f} | {r['r2_shuf']:+.4f} | {100*r['var_real']:.1f}% | {100*r['var_ctrl']:.1f}% |")
display(Markdown("**Table 1 — the iterative probe.** Each row is a probe fit on states with all previous row "
                 "spaces removed. The **random-ablation control** has the same number of dimensions removed at "
                 "each step, chosen at random from the remaining subspace, so it isolates 'we deleted position' "
                 "from 'we deleted variance'. The **shuffled-label floor** is the same probe fit against "
                 "permuted targets — where chance actually sits at this sample size.\n\n" + "\n".join(rows)))

ks = np.array([r["k"] for r in ROWS])
dims = np.array([r["dim_before"] for r in ROWS])       # x-axis: what each probe actually saw removed
dims_after = np.array([r["dim"] for r in ROWS])
C_REAL, C_CTRL, C_SHUF = "#0072B2", "#E69F00", "0.45"
plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))

ax[0].plot(dims, [r["r2"] for r in ROWS], "o-", color=C_REAL, lw=2.2, ms=4,
           label="iterative deflation (position directions removed)")
ax[0].plot(dims, [r["r2_ctrl"] for r in ROWS], "s--", color=C_CTRL, lw=2.0, ms=4,
           label="random-ablation control (same number of dims, random)")
ax[0].plot(dims, [r["r2_shuf"] for r in ROWS], ":", color=C_SHUF, lw=1.8, label="shuffled-label floor")
ax[0].axhline(0, color="0.4", ls=":", lw=1.0)
ax[0].set_xlabel("accumulated dimensions removed from h"); ax[0].set_ylabel("position R² (held-out)")
ax[0].set_title("(a) how much position survives removal", fontsize=10)
ax[0].legend(fontsize=7.5); ax[0].grid(alpha=0.3); style_ax(ax[0])

ax[1].plot(dims, [100 * r["var_real"] for r in ROWS], "o-", color=C_REAL, lw=2.2, ms=4, label="iterative deflation")
ax[1].plot(dims, [100 * r["var_ctrl"] for r in ROWS], "s--", color=C_CTRL, lw=2.0, ms=4, label="random ablation")
ax[1].plot(dims, 100 * (1 - dims / Hd), "-", color="0.6", lw=1.4,
           label=f"isotropic reference (1 − k/{Hd})")
ax[1].set_xlabel("accumulated dimensions removed from h"); ax[1].set_ylabel("state energy remaining (%)")
ax[1].set_title("(b) is R² falling because the state is gone?", fontsize=10)
ax[1].legend(fontsize=7.5); ax[1].grid(alpha=0.3); style_ax(ax[1])

ax[2].plot(ks, dims_after, "o-", color=C_REAL, lw=2.2, ms=4)
for p, c in [(0.90, "#009E73"), (0.95, "#CC79A7"), (0.99, "#D55E00")]:
    ax[2].axhline(HULL[p], color=c, ls="--", lw=1.3)
    ax[2].annotate(f"PCA hull @{int(100*p)}%: {HULL[p]} dims", xy=(ks[-1], HULL[p]), fontsize=7,
                   color=c, ha="right", va="bottom")
ax[2].set_xlabel("iteration (number of probes)"); ax[2].set_ylabel("accumulated dimensions")
ax[2].set_title("(c) size of the removed subspace, against the\ndimensions the states actually occupy", fontsize=10)
ax[2].grid(alpha=0.3); style_ax(ax[2])

fig.suptitle("Fig 1 — iterative nullspace projection of the linear position code (GRU H256)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_iterative_probing.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [5] Table 2 — the dimension is threshold-dependent, so report it at several thresholds.
r2s = np.array([r["r2"] for r in ROWS])
rows = ["| criterion | iterations (probes) | dimensions removed | % of 256 ambient | % of the 99% PCA hull | R² there |",
        "|---|---|---|---|---|---|"]
for thr in (0.50, 0.25, 0.10, 0.05, R2_STOP):
    below = np.where(r2s < thr)[0]
    if below.size == 0:
        rows.append(f"| first probe with R² < {thr:.2f} | not reached in {N_IT} | — | — | — | — |"); continue
    i = int(below[0]); r = ROWS[i]
    rows.append(f"| first probe with R² < {thr:.2f} | {r['k']} | **{r['dim_before']}** | "
                f"{100*r['dim_before']/Hd:.1f}% | {100*r['dim_before']/HULL[0.99]:.1f}% | {r['r2']:.3f} |")
display(Markdown("**Table 2 — how big is 'the position subspace'?** It depends where the threshold is put, "
                 "which is why the curve in Fig 1a is the primary result and this table is the summary. "
                 "'Accumulated dimensions' is the size of the removed subspace at the FIRST probe whose "
                 "held-out R² falls below the criterion — i.e. the code was exhausted somewhere at or below "
                 "that size.\n\n" + "\n".join(rows)))

# A single directly-comparable statement for the row-space finding this thread reports elsewhere.
first_probe = ROWS[0]
display(Markdown(
    f"**Against the single-probe picture.** One probe reads **{first_probe['rank']} of {Hd}** dimensions "
    f"(the reachable set of a readout-injection editor, chance level `√(4/256)` = {np.sqrt(4/Hd):.3f}). "
    f"The linear position code is exhausted only after removing **{TOTAL_DIM} dimensions** — "
    f"**{TOTAL_DIM // max(first_probe['rank'], 1)}× larger** — while the states themselves occupy only "
    f"**{HULL[0.99]}** dimensions at 99% variance. So the position code spans "
    f"**{100*TOTAL_DIM/HULL[0.99]:.0f}%** of the subspace the states actually live in."))

In [ ]:
# [6] Fig 2 — does position survive NONLINEARLY after its linear directions are removed?
def fit_mlp(X, Y, tr, te, steps=300, hidden=256, seed=0):
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE); Yt = torch.tensor(Y, dtype=torch.float32, device=DEVICE)
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(X.shape[1], hidden), torch.nn.ReLU(),
                              torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
                              torch.nn.Linear(hidden, Y.shape[1])).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(steps):
        opt.zero_grad(); ((net(Xt[tr]) - Yt[tr]) ** 2).mean().backward(); opt.step()
    with torch.no_grad():
        pred = net(Xt[te]).cpu().numpy()
    ss_res = ((pred - Y[te]) ** 2).sum(); ss_tot = ((Y[te] - Y[tr].mean(0)) ** 2).sum()
    return float(1 - ss_res / ss_tot)

# Evaluate the MLP on exactly the states each LINEAR probe saw, so the two curves share an x-axis,
# plus one final point after every deflation has been applied.
X_chk = H_ALL.copy(); mlp_vals = []
t0 = time.perf_counter()
for i in range(N_IT):
    mlp_vals.append(fit_mlp(X_chk, Y_ALL, TR, TE))     # states with ROWS[i]["dim_before"] dims removed
    X_chk = deflate(X_chk, DIRS[i])
mlp_vals.append(fit_mlp(X_chk, Y_ALL, TR, TE))         # after all N_IT deflations
print(f"fitted {len(mlp_vals)} MLP probes in {time.perf_counter()-t0:.0f}s")

mlp_x = np.append(dims, dims_after[-1])
mlp_curve = np.array(mlp_vals)
lin_curve = r2s

fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.6))
ax[0].plot(dims, lin_curve, "o-", color=C_REAL, lw=2.2, ms=4, label="linear probe")
ax[0].plot(mlp_x, mlp_curve, "^-", color="#009E73", lw=2.2, ms=5, label="MLP probe (2x256, ReLU)")
ax[0].axhline(0, color="0.4", ls=":", lw=1.0)
ax[0].set_xlabel("accumulated dimensions removed (all chosen by LINEAR probes)")
ax[0].set_ylabel("position R² (held-out)")
ax[0].set_title("(a) is position still there nonlinearly?", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3); style_ax(ax[0])

ax[1].plot(dims, [r["rmse"] for r in ROWS], "o-", color=C_REAL, lw=2.2, ms=4, label="iterative deflation")
ax[1].plot(dims, [r["rmse_ctrl"] for r in ROWS], "s--", color=C_CTRL, lw=2.0, ms=4, label="random ablation")
spread = float(np.sqrt(((Y_ALL[TE] - Y_ALL[TR].mean(0)) ** 2).sum(1).mean()))
ax[1].axhline(spread, color="0.35", ls=":", lw=1.4)
ax[1].annotate(f"predicting the mean: {spread:.2f}", xy=(dims[-1], spread), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[1].set_xlabel("accumulated dimensions removed from h"); ax[1].set_ylabel("position RMSE (sim units)")
ax[1].set_title("(b) the same decay in physical units", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); style_ax(ax[1])

fig.suptitle("Fig 2 — linear versus nonlinear readability of position under linear deflation", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_linear_vs_mlp.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
# Part 2 — Editing inside the 112-dimensional position subspace

Part 1 found that the linear position code spans **116 dimensions** (29 probes × rank 4; readability is dead by
the 112 removed before the last probe). Readout injection — the thread's canonical failing structural editor —
writes inside **one probe's 4 dimensions**. This part asks the obvious question: **does writing to the whole
code work?**

## The editor, and why it is exactly solvable

Probe `k` was fit on the deflated state `h_k`, but `A_k`'s rows lie in `span(B_k)`, orthogonal to every earlier
`B_j`. So `A_k h_k = A_k h` — **every probe can be read directly off the original state**. Stacking all 29 gives
`A` of shape `(116, 256)`, and because the row-space blocks are mutually orthogonal, `AAᵀ` is **block
diagonal**. The minimum-norm solution of `A Δh = δ` therefore decomposes with no interference:

$$\Delta h \;=\; \sum_k A_k^{+}\,\delta_k, \qquad \delta_k = \text{target}_k - (A_k h + b_k), \qquad A_j \Delta h = \delta_j \;\;\forall j$$

116 equations, 256 unknowns — underdetermined, solved exactly, and `Δh` lies entirely inside the 116-dim
subspace. **The multi-probe editor is literally the sum of 29 independent readout injections**, each in its own
orthogonal slice.

## Why the targets need shrinking — and why it is NOT about resolving conflicts

There is no conflict to arbitrate: every constraint is satisfiable simultaneously and exactly. The problem is
that **the constraint is false for a low-R² probe.** Probe 29 has R² = 0.02, so on a *genuine* state with the
object at the target it reads approximately the population **mean**, not the target. Demanding that it read the
target asks for a state no real state resembles.

It is also expensive. Low R² means least squares shrank `A_k` toward zero, so `σ(A_k)` is small and `A_k⁺` is
large; meanwhile `δ_k` is *bigger* for late probes, because `p_k(h)` already sits near the mean rather than near
the current position. Both compound, and since the blocks are orthogonal the norms add in quadrature, so
`‖Δh‖` ends up dominated by exactly the probes carrying the least information.

So "weighting" means **shrinking each probe's target toward what a real edited state would actually produce**:
`target_k = μ + R²_k·(target − μ)`, with `μ` the population mean position. The naive uniform version is run
anyway, so the blow-up is measured rather than asserted.

## The arms

| arm | `target_k` | what it tests |
|---|---|---|
| **unsteered** | — | the floor; this model's own `−1` end |
| **readout injection (K=1)** | target, probe 1 only | the thread's canonical failing editor, reproduced inside this notebook |
| **multi-probe uniform, K = 2…29** | target, for the first `K` probes | does writing to *more* of the position code help? |
| **multi-probe R²-shrunk (all 29)** | `μ + R²_k(target − μ)` | the same, with each probe asked for what a real edited state would give |
| **projection of the true edit, `P_S(Δh_true)`** *(oracle)* | `A_k h_cf` | **the ceiling.** See below. |
| **counterfactual state overwrite** *(oracle)* | — | the working edit the projection is derived from |

**The `P_S(Δh_true)` arm is the decisive one.** Set each probe's target to what it *actually reads on the
oracle's post-edit state*; the min-norm solution is then `Σ_k B_kB_kᵀ Δh_true = P_S(Δh_true)` — exactly the
projection of the true, working edit onto the 116-dim position subspace. So it answers: **if you keep only the
part of a successful edit that lives in the position code, does it still work?** If this fails, no
target-choosing scheme confined to this subspace can succeed, and the subspace is the wrong place to write.

> **Precedent worth holding in mind.** The tangent-constrained experiment (2026-08-05) projected the same
> working oracle onto a 22-dim local-PCA basis: it kept **57%** of `Δh_true`, cosine **+0.568**, no degradation —
> and still scored only **−0.197** Edit Index. Keeping most of the true edit bought essentially none of the
> effect. A high projection fraction does **not** imply the edit lands.

**Provenance note.** The probe cascade is fit on the `test` split (2,000 sequences, all timesteps); the editors
are applied at the edit frame of the `edits` split. That is the same pattern §4 of the thread uses. It does mean
the `K=1` arm here is not numerically identical to the §4 readout-injection row, which used a probe fit on 600
sequences — it is the same construction, refit.

In [ ]:
# [7] Load the edits split, build the oracle edit state, and the canonical §4 ray zones.
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

K_ROLL, N_EVAL = 15, 256
eb = load_dataset(f"{ROOT}/datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits = eb.edits
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res
with h5py.File(edits.h5_path, "r") as f:
    VEL_ALL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, np.float32)
COL  = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
DT   = float(sim["dt"])

def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)

def render_traj(pos_seq, noise=0.0):
    _, _, inten = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                     colors=COL, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return inten.astype(np.float32)

@torch.no_grad()
def warm(model, obs_np, upto):
    o = torch.from_numpy(obs_np).float().to(DEVICE); state = None
    for t in range(upto):
        _, state = model.step(o[:, t], state)
    return model.flat_state(state)

@torch.no_grad()
def roll(model, h_flat, steps=K_ROLL):
    """Free-run. out[:, 0] = decode(h) = sim frame ef."""
    st = model.state_from_flat(h_flat); out = [model.decode(st)]
    for _ in range(steps - 1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

IDX = np.arange(N_EVAL)
oe  = edits.edit_object[IDX].astype(int)
pos = edits.positions[IDX][:, :, :N_OBJ, :].astype(np.float32)
TGT = pos[:, ef].copy()                     # edited object already at the target; other on its true path
PRE = pos[:, ef - 1]

# counterfactual history: the edited object on a constant-velocity line ARRIVING at the target at ef
cf_obs = np.zeros((N_EVAL, ef, R), np.float32); t_idx = np.arange(ef)
for i in range(N_EVAL):
    o, other = oe[i], 1 - oe[i]
    v = VEL_ALL[IDX[i], ef, o]
    cf = np.zeros((ef, N_OBJ, 2), np.float32)
    cf[:, o]     = TGT[i, o][None, :] - v[None, :] * (ef - t_idx)[:, None] * DT
    cf[:, other] = pos[i, :ef, other]
    cf_obs[i] = render_traj(cf)

H0   = warm(MODEL, edits.obs[:N_EVAL].astype(np.float32), ef)     # pre-edit state
H_CF = warm(MODEL, cf_obs, ef)                                    # counterfactual overwrite (the working oracle)
DH_TRUE = (H_CF - H0).cpu().numpy().astype(np.float64)            # the successful edit, as a displacement

gt_roll = edits.clean_obs[:N_EVAL, ef:ef + K_ROLL, :].astype(np.float32)
ZONES = build_edit_zones(pre_pos=PRE, tgt_pos=TGT, pre_vel=VEL_ALL[IDX, ef - 1, :N_OBJ, :],
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N_EVAL, ef:ef + K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
print(f"edits split: N={N_EVAL}, ef={ef}, K_ROLL={K_ROLL}")
print(f"ray zones/sample: target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"differing {ZONES.differing.sum(1).mean():.1f}")
print(f"oracle displacement ‖Δh_true‖/‖h0‖ = "
      f"{float(np.linalg.norm(DH_TRUE,axis=-1).mean() / np.linalg.norm(H0.cpu().numpy(),axis=-1).mean()):.2f}")

In [ ]:
# [8] Fig 3 — DIAGNOSTIC, run before any editing: where does a SUCCESSFUL edit live?
# (a) how much of Δh_true lies in the first-K probes' subspace, against matched chance √(4K/256)
# (b) principal angles between the 116-dim position subspace and the decoder's 128-dim row space
S_ALL = np.concatenate([c["B"] for c in CASCADE], 1)          # (256, 116) orthonormal
n_true = np.linalg.norm(DH_TRUE, axis=-1)

frac, chance, Ks = [], [], []
for K in range(1, len(CASCADE) + 1):
    S_K = S_ALL[:, :4 * K]
    frac.append(float((np.linalg.norm(DH_TRUE @ S_K, axis=-1) / n_true).mean()))
    chance.append(float(np.sqrt(S_K.shape[1] / Hd))); Ks.append(K)
frac = np.array(frac); chance = np.array(chance); dimsK = 4 * np.array(Ks)

# decoder row space: decode is a single Linear(256 -> 128), so h splits into 128 dims that reach the
# observation immediately and 128 that only act through the recurrence.
W_dec = MODEL.decoder.weight.detach().cpu().numpy().astype(np.float64)      # (128, 256)
U_dec = np.linalg.svd(W_dec, full_matrices=False)[2].T                      # (256, 128) orthonormal row space
sv_ang = np.linalg.svd(S_ALL.T @ U_dec, compute_uv=False)
angles = np.degrees(np.arccos(np.clip(sv_ang, -1, 1)))
dec_frac = float((np.linalg.norm(DH_TRUE @ U_dec, axis=-1) / n_true).mean())

display(Markdown(
    f"**Diagnostic.** Fraction of the successful edit `Δh_true` inside the **full 116-dim position subspace**: "
    f"**{frac[-1]:.3f}** against matched chance `√(116/256)` = **{chance[-1]:.3f}** "
    f"(**{frac[-1]/chance[-1]:.2f}× chance**). For one probe it is **{frac[0]:.3f}** vs chance "
    f"**{chance[0]:.3f}** ({frac[0]/chance[0]:.2f}×).  \n"
    f"**Decoder row space.** `Δh_true` has **{dec_frac:.3f}** of its norm in the decoder's 128-dim row space "
    f"(chance `√(128/256)` = {np.sqrt(128/Hd):.3f}). Principal angles between the 116-dim position subspace and "
    f"that row space: median **{np.median(angles):.0f}°**, "
    f"{int((angles < 45).sum())}/{len(angles)} below 45° — so the position code is "
    + ("substantially aligned with the directions that reach the observation."
       if np.median(angles) < 60 else
       "largely NOT aligned with the directions that reach the observation.")))

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.5))
ax[0].plot(dimsK, frac, "o-", color="#0072B2", lw=2.2, ms=4, label="‖P·Δh_true‖ / ‖Δh_true‖")
ax[0].plot(dimsK, chance, "--", color="0.4", lw=1.6, label="chance for a random vector, √(d/256)")
ax[0].set_xlabel("dimensions of the accumulated position subspace (4 × K probes)")
ax[0].set_ylabel("fraction of the successful edit captured")
ax[0].set_title("(a) how much of a WORKING edit lives in the position code", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3); style_ax(ax[0])

ax[1].plot(dimsK, frac / chance, "o-", color="#009E73", lw=2.2, ms=4)
ax[1].axhline(1.0, color="0.4", ls="--", lw=1.4)
ax[1].annotate("chance", xy=(dimsK[-1], 1.0), fontsize=8, color="0.4", ha="right", va="bottom")
ax[1].set_xlabel("dimensions of the accumulated position subspace")
ax[1].set_ylabel("enrichment  (fraction ÷ chance)")
ax[1].set_title("(b) the same, as enrichment over chance\n(chance moves with d, so the raw fraction alone would mislead)",
                fontsize=10)
ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle("Fig 3 — is a successful edit inside the linear position code?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_edit_in_position_subspace.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [9] Build the multi-probe editors. All Δh are computed in float64, then applied to the float32 state.
h0_np = H0.cpu().numpy().astype(np.float64)
tgt4 = TGT.reshape(N_EVAL, N_OBJ * 2).astype(np.float64)
MU = Y_ALL[TR].mean(0)                                  # population mean position, from the probe-fit bank
A_PINV = [np.linalg.pinv(c["A"]) for c in CASCADE]       # (256, 4) each
R2K = np.array([c["r2"] for c in CASCADE])

def read(k, h):
    """Probe k's readout, applied directly to the ORIGINAL state (valid: A_k ⟂ every earlier B_j)."""
    return h @ CASCADE[k]["A"].T + CASCADE[k]["b"]

def multiprobe(h, K, shrink=False, target=None):
    """Σ_{k<K} A_k⁺ (target_k − p_k(h)). Exact for every probe simultaneously (orthogonal blocks)."""
    dh = np.zeros_like(h)
    for k in range(K):
        t = target[k] if target is not None else (MU + R2K[k] * (tgt4 - MU) if shrink else tgt4)
        dh += (t - read(k, h)) @ A_PINV[k].T
    return dh

K_SWEEP = [1, 2, 3, 6, 12, 20, len(CASCADE)]
EDITS_DH = {}
for K in K_SWEEP:
    EDITS_DH[f"multi-probe uniform · K={K}" if K > 1 else "readout injection · K=1 (the failing editor)"] = \
        multiprobe(h0_np, K)
EDITS_DH[f"multi-probe R²-shrunk · K={len(CASCADE)}"] = multiprobe(h0_np, len(CASCADE), shrink=True)
# the ceiling: target_k = what probe k reads on the ORACLE state  ->  Δh = P_S(Δh_true) exactly
EDITS_DH["projection of the true edit onto the position subspace (ORACLE)"] = \
    multiprobe(h0_np, len(CASCADE), target=[read(k, h0_np + DH_TRUE) for k in range(len(CASCADE))])

# verify the identity rather than asserting it
dh_proj = EDITS_DH["projection of the true edit onto the position subspace (ORACLE)"]
resid = float(np.abs(dh_proj - DH_TRUE @ S_ALL @ S_ALL.T).max())
print(f"identity check  Σ_k A_k⁺(A_k Δh_true) == P_S(Δh_true):  max abs difference {resid:.2e}  (must be ~0)")

STATES = {"unsteered (no edit)": H0}
for lab, dh in EDITS_DH.items():
    STATES[lab] = torch.from_numpy((h0_np + dh).astype(np.float32)).to(DEVICE)
STATES["counterfactual state overwrite (ORACLE)"] = H_CF

n0 = np.linalg.norm(h0_np, axis=-1)
rows = ["| editor | dims written | ‖Δh‖ / ‖h0‖ | vs the oracle's ‖Δh_true‖/‖h0‖ |", "|---|---|---|---|"]
mag_true = float((np.linalg.norm(DH_TRUE, axis=-1) / n0).mean())
MAGS = {}
for lab, dh in EDITS_DH.items():
    m = float((np.linalg.norm(dh, axis=-1) / n0).mean()); MAGS[lab] = m
    d = 4 * (len(CASCADE) if ("K=%d" % len(CASCADE)) in lab or "projection" in lab else int(lab.split("K=")[1].split(" ")[0]))
    rows.append(f"| {lab} | {d} | **{m:.2f}** | {m/mag_true:.1f}× |")
rows.append(f"| counterfactual state overwrite (ORACLE) | — (writes all 256) | {mag_true:.2f} | 1.0× |")
display(Markdown("**Table 3 — displacement size.** The uniform arms are expected to blow up: a low-R² probe has "
                 "a small `A_k`, hence a large `A_k⁺`, *and* a larger `δ_k` (its readout sits near the population "
                 "mean rather than near the current position). Orthogonal blocks means these add in quadrature. "
                 "An edit far larger than the oracle's is a warning that the state has left the manifold.\n\n"
                 + "\n".join(rows)))

In [ ]:
# [10] Table 4 + Fig 4 — the canonical §4 scorecard for every arm, at step 0 AND across the rollout.
ORDER = ["unsteered (no edit)"] + list(EDITS_DH) + ["counterfactual state overwrite (ORACLE)"]
CARDS, ROLLS = {}, {}
for lab in ORDER:
    rl = roll(MODEL, STATES[lab]); ROLLS[lab] = rl
    c = edit_scorecard(rl, ZONES, gt_roll)
    c["fidelity_ratio"] = 1.0 if lab == "unsteered (no edit)" else fidelity_ratio(c, CARDS["unsteered (no edit)"])
    CARDS[lab] = c

rows = ["| editor | Edit Index (step 0) ↑ | Edit Index (step 14) ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio ↓ |",
        "|---|---|---|---|---|---|---|---|"]
for lab in ORDER:
    c = CARDS[lab]
    rows.append(f"| {lab} | **{c['edit_index']:+.2f}** | {c['edit_index_by_step'][-1]:+.2f} | {c['target_rmse']:.3f} | "
                f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown(f"**Table 4 — does writing to the whole position code edit the world?** N={N_EVAL} held-out "
                 "edits, GRU H256. Read every index against the **unsteered** row — that is where this model's "
                 "`−1` end sits. **fidelity ratio > 1 means the edited rollout ended FURTHER from the true "
                 "post-edit world than doing nothing**, i.e. the edit degraded the model rather than steering "
                 "it; no success claim survives that.\n\n" + "\n".join(rows)))

CMAP = {"unsteered (no edit)": "0.55", "counterfactual state overwrite (ORACLE)": "#009E73",
        "projection of the true edit onto the position subspace (ORACLE)": "#CC79A7"}
def col(lab):
    if lab in CMAP: return CMAP[lab]
    if "shrunk" in lab: return "#D55E00"
    return plt.cm.Blues(0.30 + 0.62 * (K_SWEEP.index(int(lab.split("K=")[1].split(" ")[0])) / (len(K_SWEEP) - 1)))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.4))
yi = np.arange(len(ORDER))
ax[0].barh(yi, [CARDS[l]["edit_index"] for l in ORDER], 0.72, color=[col(l) for l in ORDER])
for x, t in [(1.0, "edited world"), (0.0, "equidistant"), (-1.0, "unedited world")]:
    ax[0].axvline(x, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(t, xy=(x, len(ORDER) - 0.4), fontsize=7, color="0.35", ha="center", va="top", rotation=90)
ax[0].set_yticks(yi); ax[0].set_yticklabels(ORDER, fontsize=7.5); ax[0].invert_yaxis()
ax[0].set_xlim(-1.05, 1.15); ax[0].set_xlabel("Edit Index at step 0")
ax[0].set_title("(a) does the edit land?", fontsize=10); ax[0].grid(alpha=0.3, axis="x"); style_ax(ax[0])

ax[1].barh(yi, [CARDS[l]["fidelity_ratio"] for l in ORDER], 0.72, color=[col(l) for l in ORDER])
ax[1].axvline(1.0, color="#D55E00", ls="--", lw=1.5)
ax[1].annotate("worse than doing nothing →", xy=(1.02, len(ORDER) - 0.4), fontsize=7, color="#D55E00",
               ha="left", va="top", rotation=90)
ax[1].set_yticks(yi); ax[1].set_yticklabels([]); ax[1].invert_yaxis()
ax[1].set_xlabel("fidelity ratio (GT-traj RMSE ÷ unsteered)")
ax[1].set_title("(b) did it steer the model, or damage it?", fontsize=10)
ax[1].grid(alpha=0.3, axis="x"); style_ax(ax[1])

for lab in ORDER:
    ax[2].plot(np.arange(K_ROLL), CARDS[lab]["edit_index_by_step"], color=col(lab), lw=2.0,
               ls="--" if "ORACLE" in lab else "-")
ax[2].axhline(0, color="0.4", ls=":", lw=1.0); ax[2].set_ylim(-1.05, 1.05)
ax[2].set_xlabel("rollout step s (0 = sim frame ef)"); ax[2].set_ylabel("Edit Index")
ax[2].set_title("(c) does it PERSIST? (dashed = oracles)", fontsize=10)
ax[2].grid(alpha=0.3); style_ax(ax[2])

fig.legend(handles=[Line2D([0], [0], color=col(l), lw=6, label=l) for l in ORDER],
           loc="upper center", ncol=2, fontsize=7.5, frameon=False, bbox_to_anchor=(0.5, 1.16))
fig.suptitle("Fig 4 — editing inside the 116-dimensional linear position subspace", y=1.24, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_multiprobe_edit_index.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [11] Fig 5 — observation waterfalls for the same arms (canonical spec, one helper for every waterfall).
N_CTX = 6
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
CTX = edits.obs[:N_EVAL, ef - N_CTX:ef, :].astype(np.float32)   # the NOISY frames actually teacher-forced

def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N_EVAL)])
gho_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N_EVAL)])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname, ctx, dpi=115):
    """Canonical waterfall (CLAUDE.md fixed spec): gray on dark; N_CTX NOISY context frames above a dashed
    edit line; below it EVERY column shows its OWN free-run from step 0 (which decodes sim frame `ef`).
    Deliberately NO shared teacher-forced `ef` row — only an oracle-observation editor ever sees that frame,
    and painting it into every column would hide the exact frame the §4 scorecard scores."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(2.95 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            a = axes[r][c]; a.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx[smp], col_bodies[c][smp]], 0), 0, 1)
            a.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in a.spines.values(): sp.set_edgecolor(TICK)
            a.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): a.axvline(tgt_cx[smp], color=TARGET_C, lw=1.5, alpha=0.9)
            if not np.isnan(gho_cx[smp]): a.axvline(gho_cx[smp], color=GHOST_C, ls="--", lw=1.5, alpha=0.9)
            if r == 0: a.set_title(col_titles[c], fontsize=7, color=TXT)
            if c == 0:
                a.set_ylabel(f"sample {smp}\nsim frame", fontsize=8, color=TXT)
                a.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                a.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: a.set_yticks([])
            a.set_xlabel("ray", fontsize=8, color=TXT); a.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="target location (where the object should be)"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost location (where it was before the edit)"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here — {N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = sim frame {ef}")]
    fig.legend(handles=handles, loc="upper center", ncol=1, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.885])
    fig.savefig(f"{OUT}/{fname}", dpi=dpi, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

teleport = np.linalg.norm(TGT[np.arange(N_EVAL), oe] - PRE[np.arange(N_EVAL), oe], axis=-1)
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
print("waterfall samples (largest teleports):", SAMPLES)

SHOW = ["unsteered (no edit)", "readout injection · K=1 (the failing editor)", "multi-probe uniform · K=6",
        f"multi-probe uniform · K={len(CASCADE)}", f"multi-probe R²-shrunk · K={len(CASCADE)}",
        "projection of the true edit onto the position subspace (ORACLE)",
        "counterfactual state overwrite (ORACLE)"]
titles = ["GT (sim)\nthe true post-edit world"] + [
    f"{l}\nindex {CARDS[l]['edit_index']:+.2f} · ghost {CARDS[l]['ghost_rmse']:.2f} · fidelity {CARDS[l]['fidelity_ratio']:.2f}"
    for l in SHOW]
waterfall_grid(titles, [gt_roll] + [ROLLS[l] for l in SHOW], SAMPLES,
               "Fig 5 — what each editor actually generates when writing inside the position subspace",
               "fig5_multiprobe_waterfall.png", ctx=CTX)

---
## Summary

In [ ]:
# [11] Computed summary.
print("=" * 96)
print("ITERATIVE PROBING — dimensionality of the linear position code in GRU H256")
print("=" * 96)
print(f"\nBank: {H_ALL.shape[0]:,} aligned states from {NPROBE} held-out test sequences, split by sequence.")
print(f"States occupy {HULL[0.90]}/{HULL[0.95]}/{HULL[0.99]} PCA dims at 90/95/99% variance, of {Hd} ambient.")
ranks = sorted(set(r["rank"] for r in ROWS))
print(f"\n1. IS IT '4 x number of probes'?  every probe had rank {ranks} over {N_IT} iterations -> "
      + ("YES, so total = 4 x %d = %d dims" % (N_IT, TOTAL_DIM) if ranks == [4]
         else "NO — total is the measured sum of ranks = %d dims" % TOTAL_DIM))
print(f"   Orthogonality of successive row spaces asserted at every step (max |inner| < 1e-6).")
print(f"\n2. DECAY: probe 1 R² = {r2s[0]:.3f} -> probe {N_IT} R² = {r2s[-1]:.3f} "
      f"(stop criterion R² < {R2_STOP}).")
for thr in (0.5, 0.25, 0.05):
    b = np.where(r2s < thr)[0]
    print(f"   first probe below R² {thr:<4}: iteration {ROWS[int(b[0])]['k'] if b.size else '—'}"
          + (f", {ROWS[int(b[0])]['dim_before']} dims removed" if b.size else " (not reached)"))
print(f"\n3. IS THE DECAY ABOUT POSITION OR ABOUT VARIANCE? at {TOTAL_DIM} dims removed:")
print(f"   iterative deflation  R² {ROWS[-1]['r2']:.3f}, {100*ROWS[-1]['var_real']:.1f}% of state energy left")
print(f"   random ablation      R² {ROWS[-1]['r2_ctrl']:.3f}, {100*ROWS[-1]['var_ctrl']:.1f}% of state energy left")
print(f"   -> " + ("the removal is POSITION-SPECIFIC: random removal of the same number of dimensions leaves "
                   "position readable" if ROWS[-1]["r2_ctrl"] - ROWS[-1]["r2"] > 0.3
                   else "NOT clearly position-specific — the control fell too; inspect Table 1"))
print(f"   shuffled-label floor at the last iteration: {ROWS[-1]['r2_shuf']:+.4f} (chance is 0)")
print(f"\n4. NONLINEAR SURVIVAL: MLP R² {mlp_curve[0]:.3f} (undeflated) -> {mlp_curve[-1]:.3f} "
      f"after {dims_after[-1]} linear dims removed; linear probe at {TOTAL_DIM} dims removed: {r2s[-1]:.3f}.")
print(f"   -> " + ("position is ALSO gone nonlinearly: the linear directions carried essentially all of it"
                   if mlp_curve[-1] < 0.15 else
                   "position SURVIVES nonlinearly — the linear deflation did not remove the information, "
                   "only its linearly-readable form"))
print(f"\n5. AGAINST THE SINGLE-PROBE PICTURE: one probe sees {ROWS[0]['rank']}/{Hd} dims; the full linear "
      f"position code is {TOTAL_DIM} dims,")
print(f"   = {100*TOTAL_DIM/HULL[0.99]:.0f}% of the {HULL[0.99]}-dim subspace the states actually occupy.")
print("\n" + "=" * 96)
print("figures:", sorted(os.listdir(OUT)))

# ── Part 2 ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 96)
print("PART 2 — EDITING INSIDE THE 116-DIMENSIONAL POSITION SUBSPACE")
print("=" * 96)
u = CARDS["unsteered (no edit)"]["edit_index"]
orc = CARDS["counterfactual state overwrite (ORACLE)"]["edit_index"]
proj = CARDS["projection of the true edit onto the position subspace (ORACLE)"]
print(f"\n6. IS A SUCCESSFUL EDIT EVEN IN THERE?  ‖P_S·Δh_true‖/‖Δh_true‖ = {frac[-1]:.3f} vs chance "
      f"{chance[-1]:.3f} = {frac[-1]/chance[-1]:.2f}x -> "
      + ("BELOW chance: the edit is no more aligned with the position code than a random vector"
         if frac[-1] < chance[-1] else "enriched over chance"))
print(f"\n7. THE CEILING (write oracle targets to every probe = P_S(Δh_true), the best any editor confined")
print(f"   to this subspace could do): Edit Index {proj['edit_index']:+.2f} vs unsteered {u:+.2f}, oracle {orc:+.2f}")
print(f"   -> recovers {100*(proj['edit_index']-u)/(orc-u):.0f}% of the oracle's gain from {100*frac[-1]:.0f}% of its vector,")
print(f"      with NO degradation (fidelity {proj['fidelity_ratio']:.2f}). "
      + ("The subspace is the wrong place to write." if proj["edit_index"] < 0 else "Partial success."))
best_ok = max((l for l in ORDER if "ORACLE" not in l and l != "unsteered (no edit)"
               and CARDS[l]["fidelity_ratio"] <= 1.0), key=lambda l: CARDS[l]["edit_index"])
print(f"\n8. BEST NON-DEGRADING STRUCTURAL EDITOR: {best_ok}")
print(f"   Edit Index {CARDS[best_ok]['edit_index']:+.2f} (unsteered {u:+.2f}), fidelity {CARDS[best_ok]['fidelity_ratio']:.2f}")
worst = "multi-probe uniform · K=%d" % len(CASCADE)
print(f"\n9. THE TRAP: {worst} posts Edit Index {CARDS[worst]['edit_index']:+.2f} — the best-looking structural")
print(f"   number here — but fidelity {CARDS[worst]['fidelity_ratio']:.2f}, Target RMSE {CARDS[worst]['target_rmse']:.3f} "
      f"(unsteered {CARDS['unsteered (no edit)']['target_rmse']:.3f}),")
print(f"   Ghost {CARDS[worst]['ghost_rmse']:.3f} (unsteered {CARDS['unsteered (no edit)']['ghost_rmse']:.3f}), "
      f"‖Δh‖/‖h0‖ {MAGS[worst]:.2f} vs the oracle's {mag_true:.2f}.")
print(f"   Every zone is WORSE than doing nothing: the index moved toward 0 by DEGRADING the output, not by")
print(f"   editing it — exactly the failure the Edit Index is built to read as ~0. Fig 5 shows the garbage.")
print("\n" + "=" * 96)

### Current results (updated 2026-08-05)

*From the run above: GRU `runs/controls/H256`. Part 1 — 78,000 aligned states from 2,000 held-out `test`
sequences, split by sequence. Part 2 — N=256 held-out edits from the `edits` split. This block is the only
place in the notebook where results live.*

## Part 1 — the linear position code is 116 dimensions

**Direct answer to "4 × number of probes?" — yes, verified.** Every one of the 29 probes came out **exactly
rank 4**, and each new row space was asserted orthogonal to everything already removed (max |inner product| <
1e-6). So **4 × 29 = 116 dimensions**, of which the **112** removed before the last probe are enough to kill
linear readability.

**The decay is gradual, so "the dimensionality" depends on the threshold.** R² falls 0.822 → 0.020:

| dims removed | 0 | 24 | 44 | 68 | 88 | 112 |
|---|---|---|---|---|---|---|
| position R² (held-out) | 0.822 | 0.479 | 0.236 | 0.091 | 0.049 | 0.020 |

Half the linear readability is gone after **24 dimensions** (6 probes) — the core — with a long tail to 112.

**Controls.** Removing **112 random** dimensions leaves position readable at **R² 0.767** versus **0.020** for
the 112 chosen ones, so the collapse is about *which* directions, not how many; the shuffled-label floor sits at
**−0.003**. The position directions carry **below-average** state energy (63.4% of energy retained versus the
random track's 56.9% ≈ isotropic 56.3%) — position is not written along the dominant variance axes. And 116 dims
is the size of the **linear** code, not of the information: an MLP refit on the fully deflated states still
reads position at **R² 0.544** (from 0.909).

**Scale.** The states occupy **40 / 75 / 172** PCA dims at 90/95/99% variance, so the code spans **65% of the
subspace the states actually occupy** — most of the state, not a corner. One probe sees 4 of 256; the code is
**28× larger**.

## Part 2 — editing in that subspace does not work, and the ceiling arm proves why

**A successful edit is *not* preferentially inside the position code — it is slightly *below* chance.**
`‖P_S·Δh_true‖/‖Δh_true‖` = **0.567** against matched chance `√(116/256)` = **0.673**, i.e. **0.84× chance**
(one probe: 0.098 vs 0.125, 0.78×). The 116-dim subspace does capture ~6× more of the true edit than one probe
did, but only because it is ~29× bigger — there is no enrichment.

**The decisive arm is the ceiling.** Setting every probe's target to what it reads on the oracle's post-edit
state gives, by an identity verified to 2.7e-15, exactly `P_S(Δh_true)` — the projection of the *working* edit
onto the position subspace. It is the best any editor confined to this subspace could possibly do. It scores
**−0.23** against unsteered **−0.67** and the full oracle's **+0.68**: it recovers **33% of the oracle's gain
from 57% of its vector**, with no degradation (fidelity 0.88). **So the subspace is the wrong place to write** —
no cleverer choice of targets can rescue it.

| editor | dims | ‖Δh‖/‖h0‖ | Edit Index (step 0) | at step 14 | fidelity ratio |
|---|---|---|---|---|---|
| unsteered | — | — | −0.67 | −0.47 | 1.00 |
| readout injection · K=1 | 4 | 0.09 | −0.65 | −0.46 | 1.00 |
| multi-probe uniform · K=6 | 24 | 0.22 | −0.62 | −0.44 | 1.00 |
| multi-probe uniform · K=12 | 48 | 0.48 | −0.44 | −0.29 | 0.99 |
| multi-probe uniform · K=29 | 116 | **2.63** | −0.06 | +0.01 | **1.57** |
| multi-probe R²-shrunk · K=29 | 116 | 0.51 | **−0.40** | −0.28 | **0.96** |
| `P_S(Δh_true)` *(oracle ceiling)* | 116 | 0.55 | −0.23 | −0.18 | 0.88 |
| counterfactual overwrite *(oracle)* | all 256 | 0.97 | **+0.68** | +0.44 | 0.56 |

**The `K=29` uniform arm is a trap, and it is worth dwelling on.** It posts the best-looking structural Edit
Index (**−0.06**) — but its fidelity ratio is **1.57**, its Target RMSE **0.787** against unsteered's 0.488, its
Ghost RMSE **0.774** against 0.589, its Collateral **0.610** against 0.125, and its displacement **2.63×‖h0‖**,
2.7× the oracle's. **Every zone is worse than doing nothing.** The index moved toward 0 by *degrading* the
output into something equidistant from both worlds — precisely the failure the Edit Index is designed to read as
≈0 rather than reward. Fig 5 shows the garbage directly: smeared bands and vertical striping with no coherent
object.

**Target shrinkage does exactly what it was designed to do.** `target_k = μ + R²_k(target − μ)` holds the
displacement to 0.51×‖h0‖ and fidelity to **0.96** (no damage), giving **−0.40** — the best *non-degrading*
structural arm, and a real move from −0.67. But it is still far short of editing, and Fig 5 shows a dim, smeared
object with the ghost still present.

**Nothing persists.** Fig 4c: every structural arm is flat or drifting toward 0 across the rollout, and the
ceiling arm sits flat at ≈−0.18. Only the counterfactual oracle holds a positive index (+0.68 → +0.44).

**A striking replication.** The tangent-constrained experiment (2026-08-05) projected the *same* working oracle
onto a completely different 22-dim local-PCA subspace: it kept **57%** of `Δh_true` and scored **−0.197**. Here a
116-dim subspace defined by an entirely different criterion also keeps **57%** and scores **−0.23**. Two
unrelated subspaces, the same retained fraction, the same near-total loss of effect. That is strong evidence for
the **all-or-nothing** reading: the edit does not decompose into a part that works and a part that doesn't.

**On the 116 ≈ 128 coincidence.** The decoder is a single `Linear(256 → 128)`, so `h` splits into 128 dims that
reach the observation immediately and 128 that act only through the recurrence. The position code is
**substantially aligned** with the observation-reaching half — median principal angle **41°**, 62 of 116 below
45° — and `Δh_true` has 0.635 of its norm there against chance 0.707. So position is *not* hiding in
"memory-only" directions, and that is not the explanation for why writing to it fails.

**Caveats.** Greedy removal order; arbitrary `R² < 0.02` stop; one model, one seed, position only. The probe
cascade is fit on `test` (2,000 sequences) and applied at the edit frame of `edits`, so the `K=1` arm is the same
*construction* as the thread's §4 readout injection but a different fit (600 sequences there) — it reproduces the
known negative (−0.65 vs −0.66) rather than being numerically identical. The gain sweep on α was not run: the
uniform-K sweep already spans 0.09×…2.63×‖h0‖ and shows both ends of the trade-off.